# SoccerNet GSR — one-sequence Kaggle test (`gn_state_prod_1`)

Runs the **full pipeline** (now including the `traj_refine` stage) on one `test`
sequence and the complete verification chain, following the repository's verified
procedure (`docs/KAGGLE_GUIDE.md`, established empirically on 2026-09-02).

**Session requirements**
* Accelerator **GPU T4 x2**, Internet **ON** (Settings ▸ Internet).
* If any `Ynniss/*` Hugging Face repo is private: add a Kaggle secret named `HF_TOKEN`.
* Licence: BroadTrack (EVS) is noncommercial-research, **no redistribution** — keep this
  notebook and any dataset made from its outputs **private**; never publish the built
  binary, the two TorchScript weights, or generated `broadtrack_calib/*.json`.

**Prerequisite**: the `traj_refine` + detector-switch changes must be **pushed** to
`https://github.com/Yass1223/gn_state_prod_1` — the clone cell fails fast if they are absent.

Approximate wall time (per the guide): dataset 10–55 min, BroadTrack ~20 min, jersey
venv ~15 min, pipeline ~2.5 h, metrics ~25 min — comfortably inside a 12 h session.
Disk rule: repo + `.venv` + sequence + caches + run outputs ALL on `/kaggle/tmp` (dies with the session); `/kaggle/working` receives ONLY the exported artifacts (metrics, audit, calibration, state, video) so a committed run persists them;
dataset zip, BroadTrack tree and jersey venv on `/kaggle/tmp` (large root overlay).

In [ ]:
# --- run configuration -------------------------------------------------------
SEQ = "SNGS-116"     # the guide's reference sequence (750 frames, 26 GT ids)
RUN_DET_AB = 0       # 1 = additionally run the earlier snft detector (the pipeline
                     #     default is YOLOv11L_HM) and produce a comparison (~+2.5 h)
RUN_REFINE_AB = 0    # 1 = additionally run with traj_refine disabled and compare (~+2.5 h)
CALIB_DATASET = ""   # optional: mounted frozen-calibration dataset (e.g. "/kaggle/input/gsr-calib"); empty = compute (best-of-N)

import os
os.environ["SEQ"] = SEQ
os.environ["RUN_DET_AB"] = str(RUN_DET_AB)
os.environ["RUN_REFINE_AB"] = str(RUN_REFINE_AB)
os.environ["CALIB_DATASET"] = CALIB_DATASET

# Optional HF token from Kaggle secrets (needed only if a Ynniss/* repo is private)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN exported from Kaggle secrets")
except Exception as e:
    print(f"no HF_TOKEN secret ({type(e).__name__}) — fine if all Ynniss/* repos are public")

## 1. Clone the repository (hardened) and check the required files

GitHub transiently refuses anonymous git-over-HTTPS from shared Kaggle IPs (observed in
run 3), so the clone retries 3× and falls back to the codeload tarball. The cell then
fails **here, not two hours in**, if the `traj_refine` / detector-switch files are not on
the remote yet.

In [ ]:
%%bash
set -e
mkdir -p /kaggle/tmp && cd /kaggle/tmp
REPO=Yass1223/gn_state_prod_1
if [ ! -d gn_state_prod_1 ]; then
  ok=0
  for i in 1 2 3; do
    GIT_TERMINAL_PROMPT=0 git clone --depth 1 "https://github.com/${REPO}" gn_state_prod_1 && { ok=1; break; }
    echo "clone attempt ${i} failed; retrying in 10s"; sleep 10
  done
  if [ "${ok}" != 1 ]; then
    echo "git refused; falling back to the codeload tarball"
    got=0
    for ref in main master; do
      curl -fL "https://codeload.github.com/${REPO}/tar.gz/refs/heads/${ref}" -o /tmp/repo.tgz && { got=1; break; }
    done
    [ "${got}" = 1 ] || { echo "codeload fallback failed too"; exit 1; }
    mkdir -p gn_state_prod_1
    tar -xzf /tmp/repo.tgz -C gn_state_prod_1 --strip-components=1
  fi
fi
cd gn_state_prod_1
for f in sn_gamestate/refine/traj_refine.py \
         sn_gamestate/refine/traj_refine_api.py \
         sn_gamestate/configs/modules/traj_refine/traj_refine.yaml \
         sn_gamestate/configs/modules/bbox_detector/yolo_ultralytics_snft_hm.yaml \
         sn_gamestate/track/tracklet_split.py \
         sn_gamestate/track/tracklet_split_api.py \
         sn_gamestate/configs/modules/tracklet_split/tracklet_split.yaml \
         docs/KAGGLE_GUIDE.md; do
  [ -f "${f}" ] || { echo "MISSING ${f} — push the local changes to GitHub first"; exit 1; }
done
grep -q "tracklet_split" sn_gamestate/configs/soccernet.yaml || { echo "soccernet.yaml lacks tracklet_split — push the local changes"; exit 1; }
grep -q "traj_refine" sn_gamestate/configs/soccernet.yaml || { echo "soccernet.yaml lacks traj_refine — push the local changes"; exit 1; }
echo "repository OK - traj_refine + detector switch present"


## 2. Python 3.9 venv, TrackLab patch, environment gate

The image interpreter (3.12) is never used: torch 1.13.1 publishes no wheels for it.
`boxmot==19.0.0` is installed with `--no-deps` (its metadata contradicts the project's
pins). `MPLBACKEND=Agg` is exported in **every** cell that touches the venv — Jupyter's
inline backend leaks into `%%bash` children and kills the 3.9 venv's matplotlib import.

In [ ]:
%%bash
set -e
export MPLBACKEND=Agg
cd /kaggle/tmp/gn_state_prod_1
if ! command -v uv >/dev/null 2>&1; then
  curl -LsSf https://astral.sh/uv/install.sh | sh
fi
export PATH="${HOME}/.local/bin:${PATH}"
uv python install 3.9
uv venv --clear --python 3.9 .venv
uv pip install --python .venv -e .
uv pip install --python .venv --no-deps boxmot==19.0.0
# gamestate-2025 -> 2024 patch on the installed TrackLab (wrong dataset task name otherwise)
SNGS_FILE=$(.venv/bin/python -c "import tracklab,os;print(os.path.join(os.path.dirname(tracklab.__file__),'wrappers','dataset','soccernet','soccernet_game_state.py'))")
sed -i 's/gamestate-2025/gamestate-2024/g' "${SNGS_FILE}"
echo "patched ${SNGS_FILE}"
# environment gate: 3.9.x / torch 1.13.1 / CUDA visible — refuse to continue otherwise
.venv/bin/python - <<'PY'
import sys, torch
ok = sys.version_info[:2] == (3, 9) and torch.__version__.startswith("1.13.1") and torch.cuda.is_available()
print("gate:", sys.version.split()[0], torch.__version__, "cuda", torch.cuda.is_available())
assert ok, "environment gate failed"
PY

## 3. Import preflight (seconds; every stage, including `traj_refine`)

In [ ]:
%%bash
set -e
export MPLBACKEND=Agg
cd /kaggle/tmp/gn_state_prod_1
.venv/bin/python scripts/preflight_imports.py

## 4. Dataset: download `test.zip` to `/kaggle/tmp`, extract only the target sequence

Primary: the SoccerNet (KAUST) server (2.8–15.1 MiB/s observed). Fallback on
error/empty/truncated zip: the Hugging Face mirror `SoccerNet/SN-GSR-2024` (whose cache
already sits on the large overlay). Zip members are `SNGS-XXX/...` at the root
(verified on the server zip), so a single-sequence extraction is one `unzip` pattern —
saves ~9 GB and ~30 min over a full-split extraction.

In [ ]:
%%bash
set -e
export MPLBACKEND=Agg
cd /kaggle/tmp/gn_state_prod_1
if [ -d "data/SoccerNetGS/test/${SEQ}/img1" ]; then
  echo "sequence ${SEQ} already extracted"; exit 0
fi
mkdir -p data/SoccerNetGS/test /kaggle/tmp/SoccerNetGS
ZIP=/kaggle/tmp/SoccerNetGS/gamestate-2024/test.zip
.venv/bin/python - <<'PY' || echo "SoccerNet server download failed; will try the Hugging Face mirror"
from SoccerNet.Downloader import SoccerNetDownloader
d = SoccerNetDownloader(LocalDirectory='/kaggle/tmp/SoccerNetGS')
d.downloadDataTask(task='gamestate-2024', split=['test'])
PY
if [ ! -s "${ZIP}" ] || ! .venv/bin/python -c "import zipfile; zipfile.ZipFile(r'${ZIP}').namelist()" >/dev/null 2>&1; then
  echo "falling back to Hugging Face: SoccerNet/SN-GSR-2024 test.zip"
  ZIP=$(.venv/bin/python -c "from huggingface_hub import hf_hub_download; print(hf_hub_download('SoccerNet/SN-GSR-2024','test.zip',repo_type='dataset'))")
fi
unzip -q -o "${ZIP}" "${SEQ}/*" -d data/SoccerNetGS/test
n=$(ls "data/SoccerNetGS/test/${SEQ}/img1" | wc -l)
[ -f "data/SoccerNetGS/test/${SEQ}/Labels-GameState.json" ] || { echo "labels missing"; exit 1; }
echo "extracted ${SEQ}: ${n} frames + Labels-GameState.json"


## 5. BroadTrack (native build on `/kaggle/tmp`) and the jersey venv

`BT_WEIGHTS_REPO=Ynniss/calibiration_weights` pins the calibration weights to HF
(deterministic on Kaggle; skips EVS git-lfs). everything here lives on `/kaggle/tmp` —
both big trees go to `/kaggle/tmp` (BroadTrack ~7 GB with libtorch, jersey venv ~7 GB).

In [ ]:
%%bash
set -e
export MPLBACKEND=Agg
cd /kaggle/tmp/gn_state_prod_1
BT_ROOT=/kaggle/tmp/broadtrack BT_WEIGHTS_REPO=Ynniss/calibiration_weights \
  bash scripts/setup_broadtrack.sh

In [ ]:
%%bash
set -e
export MPLBACKEND=Agg
cd /kaggle/tmp/gn_state_prod_1
JN_VENV=/kaggle/tmp/.venv_jn bash scripts/setup_jn_gsr.sh

## 6. The run — full pipeline on one sequence

Includes the new `traj_refine` stage. The overrides are exactly the guide's: the five
calibration paths under the moved `BT_ROOT`, the jersey `venv_python` under the moved
`JN_VENV`, `num_cores=0` (cuDNN/torch_shm_manager multiprocessing issue), and
`echo "n" |` for tracklab's interactive prompt. The jersey stage computes fresh (first
run of blob schema 2 in this session).

In [ ]:
%%bash
set -e
export MPLBACKEND=Agg
cd /kaggle/tmp/gn_state_prod_1
mkdir -p eval_results
# Frozen calibration: mounting a dataset of broadtrack_calib JSONs makes the
# calibration stage deterministic (use_cached_json short-circuits best-of-N).
if [ -n "${CALIB_DATASET:-}" ] && ls "${CALIB_DATASET}"/*.json >/dev/null 2>&1; then
  mkdir -p broadtrack_calib && cp "${CALIB_DATASET}"/*.json broadtrack_calib/
  echo "frozen calibration mounted: $(ls broadtrack_calib)"
fi
echo "n" | .venv/bin/tracklab -cn soccernet \
    dataset.eval_set=test dataset.nvid=1 "dataset.vids_dict.test=['${SEQ}']" \
    num_cores=0 \
    modules.calibration.cfg.binary=/kaggle/tmp/broadtrack/bin/broadtrack \
    modules.calibration.cfg.keypoint_model=/kaggle/tmp/broadtrack/models/nbjw_keypoint_model.pt \
    modules.calibration.cfg.line_model=/kaggle/tmp/broadtrack/models/tvcalib_model.pt \
    modules.calibration.cfg.libtorch_lib=/kaggle/tmp/broadtrack/libtorch/lib \
    modules.calibration.cfg.compute_tripod_script=/kaggle/tmp/broadtrack/src/scripts/compute_tripod.py \
    modules.jersey_number_detect.cfg.venv_python=/kaggle/tmp/.venv_jn/bin/python \
    2>&1 | tee eval_results/eval_test.log

## 7. Verification chain

`verify_run_integrity.py` reads the audit stage's per-sequence verdicts — which now
include the `traj_refine` check (settings/pin vs config, relabel-only, tracklet
arithmetic, merge distances ≤ tau, collisions, per-track constancy) — and exits non-zero
on any FAIL.

In [ ]:
%%bash
set -e
export MPLBACKEND=Agg
cd /kaggle/tmp/gn_state_prod_1
.venv/bin/python scripts/verify_run_integrity.py --expect-sequences 1

In [ ]:
# traj_refine: what the new stage actually did on this sequence (audit sidecar)
import json, os
p = f"/kaggle/tmp/gn_state_prod_1/audit/traj_refine/{os.environ['SEQ']}.json"
d = json.load(open(p))
print(json.dumps({k: d[k] for k in ("settings", "inputs", "outputs") if k in d}, indent=2))
print("\nmerged clusters:")
print(json.dumps(d.get("per_cluster", []), indent=2))
print("\nconflicts:")
print(json.dumps((d.get("phase2a") or {}).get("conflict_log", []), indent=2))

In [ ]:
%%bash
set -e
export MPLBACKEND=Agg
cd /kaggle/tmp/gn_state_prod_1
STATE=$(ls -t outputs/sn-gamestate/*/*/states/sn-gamestate.pklz | head -1)
echo "state: ${STATE}"
.venv/bin/python scripts/reference_metrics.py --state "${STATE}" \
    --dataset-path data/SoccerNetGS --eval-set test --out reference_metrics

In [ ]:
%%bash
set -e
export MPLBACKEND=Agg
cd /kaggle/tmp/gn_state_prod_1
.venv/bin/python scripts/verify_broadtrack_conversion.py \
    -c broadtrack_calib/${SEQ}.json \
    -f data/SoccerNetGS/test/${SEQ}/img1 \
    -o broadtrack_check

## 8. Export the run artifacts to `/kaggle/working`

Everything heavy (repo, venvs, dataset, caches, run dirs) lives on `/kaggle/tmp` and
dies with the session. `/kaggle/working` receives ONLY the artifacts worth keeping --
metrics, the audit verdict + every stage sidecar, the calibration JSON (a candidate
for the frozen-calibration dataset), the TrackLab state, the visualization video and
the jersey cache -- so a committed run (Save & Run All) persists them in the Output.


In [ ]:
%%bash
set -e
cd /kaggle/tmp/gn_state_prod_1
OUT=/kaggle/working
RUN_DIR=$(ls -dt outputs/sn-gamestate/*/*/ | head -1)
echo "run dir: ${RUN_DIR}"
mkdir -p "${OUT}/eval_results" "${OUT}/audit" "${OUT}/calibration" "${OUT}/states" "${OUT}/video"
cp -r eval_results/. "${OUT}/eval_results/" 2>/dev/null || true
cp -r audit/. "${OUT}/audit/" 2>/dev/null || true                  # verdicts + every stage sidecar
cp -r broadtrack_calib/. "${OUT}/calibration/" 2>/dev/null || true # keep the draw: freeze candidate
find "${RUN_DIR}states" -name '*.pklz' -exec cp {} "${OUT}/states/" \; 2>/dev/null || true
find "${RUN_DIR}" -path '*visualization*' \( -name '*.mp4' -o -name '*.avi' -o -name '*.mkv' \) -exec cp {} "${OUT}/video/" \; 2>/dev/null || true
if [ -d jn_cache ]; then (cd jn_cache && zip -qr "${OUT}/jn_cache.zip" .); fi  # reusable across sessions
echo; echo '--- /kaggle/working after export ---'
du -sh "${OUT}"/* 2>/dev/null || true
N_VID=$(ls "${OUT}/video" 2>/dev/null | wc -l)
if [ "${N_VID}" -lt 1 ]; then echo 'WARNING: no visualization video found'; find "${RUN_DIR}" -maxdepth 2 -type d; fi


Reference points from the pre-`traj_refine` runs on this sequence (2026-09-02,
two sessions): in-run GS-HOTA 58.4–59.4; reference tracking HOTA 64.85 / MOTA 86.90 /
IDF1 80.57; GS-HOTA 59.45; jersey F1 0.806, trk_acc 0.857; calibration JaC5 0.481,
MRE 4.74 px. Single-sequence numbers — not comparable to full-split figures. The delta
against these is the `traj_refine` contribution (same detector, same sequence).

## 9. Optional A/B runs (off by default — each adds ~2.5 h)

Enable via `RUN_DET_AB` / `RUN_REFINE_AB` in the configuration cell.

* **Detector A/B**: the session's main run uses the default detector (YOLOv11L_HM);
  this rerun selects the earlier fine-tune (`modules/bbox_detector=yolo_ultralytics_snft`),
  same operating point, different weights. The calibration cache
  (`broadtrack_calib/`) is deliberately **reused** so both runs share camera
  parameters — that isolates the detector's effect. The jersey cache re-keys on the
  new boxes automatically.
* **traj_refine off**: `modules.traj_refine.cfg.enabled=false` — snapshots + sidecar
  only, detections unchanged; attribution of the refine stage on identical inputs.

In [ ]:
%%bash
set -e
export MPLBACKEND=Agg
cd /kaggle/tmp/gn_state_prod_1
if [ "${RUN_DET_AB:-0}" != "1" ]; then echo "detector A/B skipped (set RUN_DET_AB = 1 in the config cell)"; exit 0; fi
echo "n" | .venv/bin/tracklab -cn soccernet \
    modules/bbox_detector=yolo_ultralytics_snft \
    dataset.eval_set=test dataset.nvid=1 "dataset.vids_dict.test=['${SEQ}']" \
    num_cores=0 \
    modules.calibration.cfg.binary=/kaggle/tmp/broadtrack/bin/broadtrack \
    modules.calibration.cfg.keypoint_model=/kaggle/tmp/broadtrack/models/nbjw_keypoint_model.pt \
    modules.calibration.cfg.line_model=/kaggle/tmp/broadtrack/models/tvcalib_model.pt \
    modules.calibration.cfg.libtorch_lib=/kaggle/tmp/broadtrack/libtorch/lib \
    modules.calibration.cfg.compute_tripod_script=/kaggle/tmp/broadtrack/src/scripts/compute_tripod.py \
    modules.jersey_number_detect.cfg.venv_python=/kaggle/tmp/.venv_jn/bin/python \
    2>&1 | tee eval_results/eval_test_snft.log
# comparison: oldest state of the session = the baseline run (default detector = HM,
# refine on); newest = the snft rerun that just finished.
BASE=$(ls -tr outputs/sn-gamestate/*/*/states/sn-gamestate.pklz | head -1)
NEW=$(ls -t outputs/sn-gamestate/*/*/states/sn-gamestate.pklz | head -1)
.venv/bin/python scripts/reference_metrics.py \
    --state "${BASE}" --label hm \
    --state "${NEW}" --label snft \
    --dataset-path data/SoccerNetGS --eval-set test --out reference_metrics_det_ab

In [ ]:
%%bash
set -e
export MPLBACKEND=Agg
cd /kaggle/tmp/gn_state_prod_1
if [ "${RUN_REFINE_AB:-0}" != "1" ]; then echo "traj_refine A/B skipped (set RUN_REFINE_AB = 1 in the config cell)"; exit 0; fi
echo "n" | .venv/bin/tracklab -cn soccernet \
    modules.traj_refine.cfg.enabled=false \
    dataset.eval_set=test dataset.nvid=1 "dataset.vids_dict.test=['${SEQ}']" \
    num_cores=0 \
    modules.calibration.cfg.binary=/kaggle/tmp/broadtrack/bin/broadtrack \
    modules.calibration.cfg.keypoint_model=/kaggle/tmp/broadtrack/models/nbjw_keypoint_model.pt \
    modules.calibration.cfg.line_model=/kaggle/tmp/broadtrack/models/tvcalib_model.pt \
    modules.calibration.cfg.libtorch_lib=/kaggle/tmp/broadtrack/libtorch/lib \
    modules.calibration.cfg.compute_tripod_script=/kaggle/tmp/broadtrack/src/scripts/compute_tripod.py \
    modules.jersey_number_detect.cfg.venv_python=/kaggle/tmp/.venv_jn/bin/python \
    2>&1 | tee eval_results/eval_test_norefine.log
BASE=$(ls -tr outputs/sn-gamestate/*/*/states/sn-gamestate.pklz | head -1)
NEW=$(ls -t outputs/sn-gamestate/*/*/states/sn-gamestate.pklz | head -1)
.venv/bin/python scripts/reference_metrics.py \
    --state "${BASE}" --label refine_on \
    --state "${NEW}" --label refine_off \
    --dataset-path data/SoccerNetGS --eval-set test --out reference_metrics_refine_ab